In [2]:
import time
import re
import os
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException,
    UnexpectedAlertPresentException, NoAlertPresentException
)

# ==============================
# 🔧 파일 경로
# ==============================
INPUT_URL_XLSX = "yeoshinticket_event_urls_from_clean.xlsx"
OUTPUT_CSV = "yeoti_price_recrawl.csv"
ERROR_CSV = "yeoti_price_recrawl_errors.csv"

SAVE_INTERVAL = 50
MAX_RETRY = 3


# ==============================
# 🔧 크롬드라이버 설정
# ==============================
def init_driver():
    chromedriver_autoinstaller.install()
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(options=options)
    return driver


# ==============================
# 🔧 단일 텍스트 추출
# ==============================
def safe_text(driver, selector):
    try:
        t = driver.find_element(By.CSS_SELECTOR, selector).text.strip()
        return t
    except:
        return ""


def extract_digits(text):
    """숫자만 추출"""
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


# ==============================
# 🔥 시술 상세 페이지 가격 추출
# ==============================
def extract_prices(driver):
    """
    정상가(original_price)
    판매가(selling_price)
    → 할인율 계산 포함
    """

    # 정상가
    SEL_ORIG = (
        "#ct-view > div.relative.w-full > div.relative.bg-white > "
        "div.px-\\[16px\\] > div.flex.flex-col.justify-center.w-full.py-\\[16px\\] "
        "> section.flex.items-end.justify-between.w-full.mt-\\[8px\\] "
        "> div > div:nth-child(1)"
    )

    # 판매가(할인 있는 경우)
    SEL_SELL_DISCOUNT = (
        "#ct-view > div.relative.w-full > div.relative.bg-white > "
        "div.px-\\[16px\\] > div.flex.flex-col.justify-center.w-full.py-\\[16px\\] "
        "> section.flex.items-end.justify-between.w-full.mt-\\[8px\\] "
        "> div > div > h2"
    )

    # 판매가(할인 없는 경우)
    SEL_SELL_NODISCOUNT = (
        "#ct-view > div.relative.w-full > div.relative.bg-white > "
        "div.px-\\[16px\\] > div.flex.flex-col.justify-center.w-full.py-\\[16px\\] "
        "> section.flex.items-end.justify-between.w-full.mt-\\[8px\\] "
        "> div > h2"
    )

    # 원가
    orig_raw = safe_text(driver, SEL_ORIG)
    original_price = extract_digits(orig_raw)

    # 판매가(두 가지 셀렉터 순서대로 시도)
    selling_raw = safe_text(driver, SEL_SELL_DISCOUNT)
    if not selling_raw:
        selling_raw = safe_text(driver, SEL_SELL_NODISCOUNT)
    selling_price = extract_digits(selling_raw)

    # 할인율 계산
    discount_rate = None
    if original_price and selling_price and original_price > selling_price:
        discount_rate = int((1 - (selling_price / original_price)) * 100)

    return original_price, selling_price, discount_rate


# ==============================
# 🔄 단일 URL 크롤링
# ==============================
def crawl_price(driver, url):
    driver.get(url)

    # 페이지 로딩 확인
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "#ct-view"))
    )

    original_price, selling_price, discount_rate = extract_prices(driver)

    return {
        "event_url": url,
        "원가": original_price,
        "할인가격": selling_price,
        "할인율": discount_rate,
    }


# ==============================
# 🔄 재시도 포함
# ==============================
def crawl_with_retry(driver, url):
    last_err = None

    for attempt in range(1, MAX_RETRY + 1):
        try:
            return crawl_price(driver, url)
        except Exception as e:
            last_err = e
            try:
                driver.switch_to.alert.dismiss()
            except:
                pass
            time.sleep(1)

    raise last_err


# ==============================
# 🧠 메인 실행
# ==============================
def main():
    df = pd.read_excel(INPUT_URL_XLSX)

    print(f"총 URL 수: {len(df)}개")

    results = []

    # 재시작 관련
    if os.path.exists(OUTPUT_CSV):
        done = pd.read_csv(OUTPUT_CSV)
        done_urls = set(done["event_url"])
        df = df[~df["event_url"].isin(done_urls)].reset_index(drop=True)
        results.extend(done.to_dict("records"))
        print(f"[재시작] 기존 {len(done)}행 로드됨.")

    driver = init_driver()

    errors = []

    try:
        for idx, row in df.iterrows():
            url = row["event_url"]
            print(f"[{idx+1}/{len(df)}] → {url}")

            try:
                rec = crawl_with_retry(driver, url)
                results.append(rec)
            except Exception as e:
                print(f"❌ 실패: {url}")
                errors.append({"event_url": url, "error": str(e)})

            # 중간 저장
            if (idx + 1) % SAVE_INTERVAL == 0:
                pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
                pd.DataFrame(errors).to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")
                print(f"[중간 저장 완료] {len(results)}건")

    finally:
        driver.quit()

    # 최종 저장
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame(errors).to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")

    print("\n===============================")
    print("🔥 재수집 완료")
    print(f"총 수집: {len(results)}건")
    print(f"실패 URL: {len(errors)}건 → {ERROR_CSV}")
    print("===============================")


if __name__ == "__main__":
    main()


총 URL 수: 3437개
[재시작] 기존 150행 로드됨.
[1/3287] → https://www.yeoshin.co.kr/event/mobile/20594
[2/3287] → https://www.yeoshin.co.kr/event/mobile/17287
[3/3287] → https://www.yeoshin.co.kr/event/mobile/9269
[4/3287] → https://www.yeoshin.co.kr/event/mobile/7227
[5/3287] → https://www.yeoshin.co.kr/event/mobile/26026
[6/3287] → https://www.yeoshin.co.kr/event/mobile/26025
[7/3287] → https://www.yeoshin.co.kr/event/mobile/26159
[8/3287] → https://www.yeoshin.co.kr/event/mobile/27534
[9/3287] → https://www.yeoshin.co.kr/event/mobile/25256
[10/3287] → https://www.yeoshin.co.kr/event/mobile/6836
[11/3287] → https://www.yeoshin.co.kr/event/mobile/17247
[12/3287] → https://www.yeoshin.co.kr/event/mobile/23065
[13/3287] → https://www.yeoshin.co.kr/event/mobile/1879
[14/3287] → https://www.yeoshin.co.kr/event/mobile/14763
[15/3287] → https://www.yeoshin.co.kr/event/mobile/16381
[16/3287] → https://www.yeoshin.co.kr/event/mobile/17280
[17/3287] → https://www.yeoshin.co.kr/event/mobile/26133
[18/3287] 